# Module 3: High-Performance AI Backend
## Task 3: The "Live" API (FastAPI + Mistral)

### **Goal**
In this notebook, we wrap our Mistral Agent into a production-ready API.
1. **Pydantic Schemas:** Define valid request and response formats.
2. **Streaming Endpoint:** Implement Server-Sent Events (SSE) for token-by-token generation.
3. **Async Execution:** Use `async/await` to handle concurrent users.

In [3]:
# !pip install fastapi uvicorn pydantic langchain-mistralai

import os
import json
import asyncio
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from typing import List, Dict, Any, AsyncIterable

# Corrected Imports for 2026 Unified Agent
from langchain_mistralai import ChatMistralAI
from langchain.agents import create_agent

# Initialize API and Model
app = FastAPI(title="Industrial AI Agent API")
os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY", "your-mistral-key-here")
llm = ChatMistralAI(model="mistral-large-latest", streaming=True)

c:\Users\zarya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\zarya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\zarya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## Data Validation (Pydantic)

In [4]:
class ChatRequest(BaseModel):
    """Schema for incoming user requests"""
    message: str = Field(..., example="What is the weather in Peshawar?")
    session_id: str = Field(default="default-session", description="Thread ID for history")

class ChatResponse(BaseModel):
    """Schema for standard (non-streaming) responses"""
    output: str

C:\Users\zarya\AppData\Local\Temp\ipykernel_12952\2731387861.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'example'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  message: str = Field(..., example="What is the weather in Peshawar?")


## The Streaming Generator Logic

In [5]:
# We reuse the agent logic from Module 2
agent = create_agent(
    model=llm,
    tools=[], # Add your tools from Module 2 here
    system_prompt="You are a high-performance industrial AI assistant."
)

async def event_generator(user_message: str) -> AsyncIterable[str]:
    """Generates Server-Sent Events (SSE) for token-by-token streaming"""
    
    # Mistral 2026 uses the 'messages' state format
    input_payload = {
        "messages": [{"role": "user", "content": user_message}]
    }

    # .astream() yields events from the LangGraph agent
    async for event in agent.astream(input_payload, stream_mode="messages"):
        # We look specifically for the message content updates
        if event and len(event) > 0:
            token = event[0].content
            if token:
                # SSE format: data: <payload>\n\n
                yield f"data: {json.dumps({'token': token})}\n\n"
        
        await asyncio.sleep(0.01) # Prevent CPU spiking

## The API Endpoints

In [6]:
@app.get("/")
def read_root():
    return {"status": "Online", "docs": "/docs"}

@app.post("/stream")
async def stream_chat(request: ChatRequest):
    """Endpoint for real-time AI generation"""
    return StreamingResponse(
        event_generator(request.message), 
        media_type="text/event-stream"
    )

### **How to Run This Service**
To launch this server, save the code as `main.py` and run the following command in your terminal:

```bash
uvicorn main:app --reload